In [1]:
import logging
import os
import random
from pathlib import Path

import openslide
import pyvips
import torch

import matplotlib.pyplot as plt
import hydra
import mlflow

import numpy as np
import pandas as pd
from lightning import seed_everything
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from explainability.cams import grad_cam, grad_cam_pp, layer_cam
from explainability.cams.abstract import (
    HookedModule,
    modify_ReLU_inplace,
    reshape_for_clustering,
)
from explainability.precomputing import ClusteringManager
from prostate_cancer.data import DataModule
from prostate_cancer.prostate_cancer_model import ProstateCancerModel

In [2]:
def get_clustering_instance_fp(clustering_algorithm: str, num_clusters: int) -> Path:
    return Path(
        "/mnt/projects/explainability/XAICNNEmbeddings/PRECOMPUTED",
        "VGG16_Prostate",
        "global_clustering_instances",
        f"clustering_instance_test_global_{clustering_algorithm.lower()}_{num_clusters}clusters.npy",
    )

In [3]:
# Define clustering algorithms and cluster numbers to test
CLUSTERING_ALGORITHMS = ["NMF", "KMeans"]
CLUSTER_NUMBERS = [4, 6, 8, 10, 12, 14]

In [4]:
# Load all clustering models
print("Loading all clustering models...")
clustering_models = {}

for clustering_algorithm in CLUSTERING_ALGORITHMS:
    for num_clusters in CLUSTER_NUMBERS:
        model_name = f"{clustering_algorithm}_{num_clusters}"
        print(f"Loading {model_name}...")

        clustering_instance_fp = get_clustering_instance_fp(
            clustering_algorithm, num_clusters
        )
        clustering_model = ClusteringManager.load_model(
            algorithm=clustering_algorithm,
            num_clusters=num_clusters,
            path=clustering_instance_fp,
        )

        clustering_models[model_name] = (
            clustering_model,
            num_clusters,
            clustering_algorithm,
        )

print(f"\nLoaded {len(clustering_models)} clustering models")
print(f"Models: {list(clustering_models.keys())}\n")

Loading all clustering models...
Loading NMF_4...
Loading NMF_6...
Loading NMF_8...
Loading NMF_10...
Loading NMF_12...
Loading NMF_14...
Loading KMeans_4...
Loading KMeans_6...
Loading KMeans_8...
Loading KMeans_10...
Loading KMeans_12...
Loading KMeans_14...

Loaded 12 clustering models
Models: ['NMF_4', 'NMF_6', 'NMF_8', 'NMF_10', 'NMF_12', 'NMF_14', 'KMeans_4', 'KMeans_6', 'KMeans_8', 'KMeans_10', 'KMeans_12', 'KMeans_14']



In [5]:
logging.basicConfig(level=logging.INFO)

# Set random seed for reproducibility
seed_everything(42, workers=True)
torch.set_float32_matmul_precision(precision="medium")

# Configuration overrides for prediction
overrides = ["experiment=predict/images/vgg16", "mode=predict"]

# Initialize Hydra configuration
with hydra.initialize(config_path="conf", version_base=None):
    config = hydra.compose(config_name="default", overrides=overrides)

print("Configuration loaded successfully!")
print(f"Mode: {config.mode}")
print(f"Checkpoint: {config.checkpoint}")
print(f"Batch size: {config.data.batch_size}")

# Instantiate data module and model
data: DataModule = hydra.utils.instantiate(
    config.data,
    _recursive_=False,  # to avoid instantiating all the datasets
    _target_=DataModule,
)

chkcpt_path = mlflow.artifacts.download_artifacts(config.checkpoint)
checkpoint = torch.load(chkcpt_path, map_location="cuda:0")

model: ProstateCancerModel = hydra.utils.instantiate(config.model)
model.load_state_dict(checkpoint["state_dict"], strict=True)

model = model.to("cuda:0")

print("Data module and model instantiated successfully!")

print(model)

Seed set to 42


Configuration loaded successfully!
Mode: predict
Checkpoint: mlflow-artifacts:/65/7b52930515c14710855962f8882fb4d3/artifacts/checkpoints/epoch=1-step=11620/checkpoint.ckpt
Batch size: 74


Data module and model instantiated successfully!
ProstateCancerModel(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
 

In [6]:
mlflow_client = MlflowClient(tracking_uri="http://mlflow.rationai-mlflow:5000/")
chkcpt_path = mlflow_client.download_artifacts(
    run_id="199141b76f90457fbf6183bd70a1e488", path="annotation_masks"
)

In [7]:
def get_carcinoma_mask(slide_id: str) -> np.ndarray | None:
    if not os.path.exists(f"{chkcpt_path}/carcinoma/{slide_id}.tiff"):
        return None

    carcinoma = pyvips.Image.new_from_file(f"{chkcpt_path}/carcinoma/{slide_id}.tiff")
    carcinoma_np = np.asarray(carcinoma) > 0

    # if exclude does not exists, it is empty
    if not os.path.exists(f"{chkcpt_path}/exclude/{slide_id}.tiff"):
        exclude_np = np.zeros_like(carcinoma_np)
    else:
        exclude = pyvips.Image.new_from_file(f"{chkcpt_path}/exclude/{slide_id}.tiff")
        exclude_np = np.asarray(exclude) > 0

    return carcinoma_np & ~exclude_np


def get_another_pathology_mask(slide_id: str) -> np.ndarray | None:
    if not os.path.exists(f"{chkcpt_path}/another_pathology/{slide_id}.tiff"):
        return None

    another_pathology = pyvips.Image.new_from_file(
        f"{chkcpt_path}/another_pathology/{slide_id}.tiff"
    )
    return np.asarray(another_pathology) > 0


def get_tile_mask(mask: np.ndarray | None, x: int, y: int) -> np.ndarray | None:
    if mask is None:
        return None
    xx, yy = x // 16, y // 16
    return mask[xx : xx + 32, yy : yy + 32]

In [8]:
target_layer = "backbone.29"
hooked_model = HookedModule(model, layer_names=[target_layer])
modify_ReLU_inplace(hooked_model, inplace=False)

In [9]:
# Get one batch from validation dataset
data.batch_size = 32
data.setup("test")
dataloaders = data.test_dataloader()

In [10]:
carcinoma_masks = {}
another_pathology_masks = {}

for i in range(len(dataloaders)):
    slide_metadata = data.test.slides.iloc[i]
    slide_id = Path(slide_metadata["path"]).stem

    carcinoma_masks[slide_id] = get_carcinoma_mask(slide_id)
    another_pathology_masks[slide_id] = get_another_pathology_mask(slide_id)

INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers
INFO:pyvips:VIPS: threadpool completed with 4 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 9 workers
INFO:pyvips:VIPS: threadpool completed with 6 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers
INFO:pyvips:VIPS: threadpool completed with 5 workers
INFO:pyvips:VIPS: threadpool

In [11]:
def create_data(clustering_models, tile_samples):
    all_datasets = {model_name: [] for model_name in clustering_models}

    # Process all tile samples
    for slide_index, tile_idx in tile_samples:
        print(f"Slide {slide_index}, Tile {tile_idx}")
        dataloader = dataloaders[slide_index]
        dataset = dataloader.dataset
        print(len(dataset), "index is", tile_idx)
        tile, label, metadata = dataset[tile_idx]
        print("Label is", label)
        tile = tile.to("cuda:0").unsqueeze(0)

        # Forward pass - compute once for all models
        output = hooked_model(tile)
        output.backward()

        # Get activations and gradients once
        activations = hooked_model.get_activations(target_layer)
        activations_flattened, shape = reshape_for_clustering(activations)
        activations_np = activations_flattened.detach().cpu().numpy()
        gradients = hooked_model.get_gradients(target_layer)

        slide, x, y = metadata["slide"], metadata["x"], metadata["y"]
        carcinoma_mask = get_tile_mask(carcinoma_masks[slide], x, y)
        another_pathology_mask = get_tile_mask(another_pathology_masks[slide], x, y)

        # Compute CAMs once (same for all models)
        cams = {}
        for cam_name, cam_fn in [
            ("grad_cam_pp", grad_cam_pp),
            ("grad_cam", grad_cam),
            ("layer_cam", layer_cam),
        ]:
            cam = cam_fn(activations, gradients)
            cam_np = cam.flatten().detach().cpu().numpy()
            cam_mask = (cam > 1e-9).detach().cpu().numpy()
            cams[cam_name] = {
                "cam": cam,
                "cam_np": cam_np,
                "cam_mask": cam_mask,
                "cam_count": cam_mask.sum(),
            }

        batch_size, _, height, width = shape

        # Process each clustering model
        for model_name, (
            clustering_model,
            num_clusters,
            clustering_algorithm,
        ) in clustering_models.items():
            # Build header for this specific model (only columns for its num_clusters)
            header = ["label", "output", "clustering_algorithm", "num_clusters"]

            # Add columns for this model's clusters
            for i in range(num_clusters):
                header.append(f"weight_sum_{i}")
                header.append(f"cluster_count_{i}")
                header.append(f"cluster_soft_count_{i}")

            # Annotation mask presence flags
            header += ["has_carcinoma_mask", "has_another_pathology_mask"]

            # CAM columns
            for cam in ["grad_cam_pp", "grad_cam", "layer_cam"]:
                header += [f"{cam}_count"]
                for i in range(num_clusters):
                    header += [
                        f"intersection_count_{cam}_{i}",
                        f"union_count_{cam}_{i}",
                    ]
                    header += [
                        f"intersection_soft_count_{cam}_{i}",
                        f"union_soft_count_{cam}_{i}",
                    ]
                    header += [f"correlation_{cam}_{i}"]
                    header += [f"l2_distance_{cam}_{i}"]

                # CAM with annotation masks
                header += [
                    f"intersection_count_{cam}_carcinoma",
                    f"union_count_{cam}_carcinoma",
                    f"intersection_count_{cam}_another_pathology",
                    f"union_count_{cam}_another_pathology",
                ]

            # Cluster masks with annotation masks
            for i in range(num_clusters):
                header += [
                    f"intersection_count_cluster_{i}_carcinoma",
                    f"union_count_cluster_{i}_carcinoma",
                    f"intersection_soft_count_cluster_{i}_carcinoma",
                    f"union_soft_count_cluster_{i}_carcinoma",
                    f"intersection_count_cluster_{i}_another_pathology",
                    f"union_count_cluster_{i}_another_pathology",
                    f"intersection_soft_count_cluster_{i}_another_pathology",
                    f"union_soft_count_cluster_{i}_another_pathology",
                ]

            # Transform activations with this clustering model
            if clustering_algorithm == "KMeans":
                # For KMeans, transform returns distances, use argmin to get cluster assignments
                distances = clustering_model.transform(activations_np)
                clustering_flat = distances.argmin(
                    axis=1
                )  # Cluster with minimum distance
                clustering = clustering_flat.reshape(batch_size, height, width)

                # For KMeans, we use distances as "weights" for sum calculation
                # but they represent distances, not probabilities
                weights_sum = np.sum(distances, axis=0)

                # For soft masks with KMeans, we could use inverse distances or just use hard masks
                # Using inverse distances normalized by sum for soft assignment
                epsilon = 1e-9
                inv_distances = 1.0 / (distances + epsilon)
                inv_distances_norm = inv_distances / (
                    inv_distances.sum(axis=1, keepdims=True) + epsilon
                )
                weights_reshaped = inv_distances_norm.reshape(
                    batch_size, height, width, num_clusters
                )
            else:
                # For NMF, transform returns weights/coefficients
                weights = clustering_model.transform(activations_np)
                weights_sum = np.sum(weights, axis=0)
                weights_reshaped = weights.reshape(
                    batch_size, height, width, num_clusters
                )

                clustering = weights.argmax(axis=1)
                clustering = clustering.reshape(batch_size, height, width)

            # Count number of tiles per cluster
            cluster_counts = np.bincount(clustering.flatten(), minlength=num_clusters)

            # Start building row data
            tile_data = [
                label.item(),
                output.cpu().item(),
                clustering_algorithm,
                num_clusters,
            ]

            # Add weight_sum, cluster_count, cluster_soft_count for this model's clusters
            for i in range(num_clusters):
                tile_data.append(weights_sum[i])
                tile_data.append(cluster_counts[i])
                if clustering_algorithm == "KMeans":
                    # For KMeans, count pixels where this cluster has highest inverse distance (soft assignment)
                    max_cluster = weights_reshaped[0, :, :, :].argmax(axis=2)
                    tile_data.append((max_cluster == i).sum())
                else:
                    # For NMF, count pixels where weight > threshold
                    tile_data.append((weights[:, i] > 1e-9).sum())

            # Add annotation mask presence flags
            has_carcinoma = carcinoma_mask is not None
            has_another_pathology = another_pathology_mask is not None
            tile_data.append(int(has_carcinoma))
            tile_data.append(int(has_another_pathology))

            # Create masks for this model
            masks = []
            for i in range(num_clusters):
                mask = np.zeros((height, width), dtype=np.float32)
                mask[clustering[0, :, :] == i] = 1
                assert mask.sum() == cluster_counts[i]
                assert mask.shape == (height, width)
                masks.append(mask)

            soft_masks = []
            for i in range(num_clusters):
                mask = np.zeros((height, width), dtype=np.float32)
                if clustering_algorithm == "KMeans":
                    # For KMeans, use inverse distance threshold for soft assignment
                    mask[weights_reshaped[0, :, :, i] > 1e-3] = 1
                else:
                    # For NMF, use weight threshold
                    mask[weights_reshaped[0, :, :, i] > 1e-9] = 1
                assert mask.shape == (height, width)
                soft_masks.append(mask)

            # Process CAMs
            for cam_name in ["grad_cam_pp", "grad_cam", "layer_cam"]:
                cam_data = cams[cam_name]
                cam_count = cam_data["cam_count"]
                cam_mask = cam_data["cam_mask"]
                cam_np = cam_data["cam_np"]

                tile_data.append(cam_count)

                for i in range(num_clusters):
                    # Use masks for this cluster
                    for cluster_mask in [masks[i], soft_masks[i]]:
                        intersection_count = (cam_mask * cluster_mask).sum()
                        union_count = (cam_mask + cluster_mask).sum()
                        tile_data.extend([int(intersection_count), int(union_count)])

                    # Correlation
                    weights_cluster = weights_reshaped[0, :, :, i].flatten()
                    if (
                        cam_count == 0
                        or np.std(cam_np) == 0
                        or np.std(weights_cluster) == 0
                    ):
                        correlation = np.nan
                    else:
                        correlation = np.corrcoef(cam_np, weights_cluster)[0, 1]
                    tile_data.append(correlation)

                    # Calculate L2 distance
                    weights_cluster_flat = weights_reshaped[0, :, :, i].flatten()
                    l2_distance = np.sqrt(np.sum((cam_np - weights_cluster_flat) ** 2))
                    tile_data.append(l2_distance)

                # CAM with annotation masks
                if has_carcinoma:
                    intersection_cam_carcinoma = (cam_mask * carcinoma_mask).sum()
                    union_cam_carcinoma = (cam_mask + carcinoma_mask).sum()
                else:
                    intersection_cam_carcinoma = 0
                    union_cam_carcinoma = 0
                tile_data.extend(
                    [int(intersection_cam_carcinoma), int(union_cam_carcinoma)]
                )

                if has_another_pathology:
                    intersection_cam_another = (cam_mask * another_pathology_mask).sum()
                    union_cam_another = (cam_mask + another_pathology_mask).sum()
                else:
                    intersection_cam_another = 0
                    union_cam_another = 0
                tile_data.extend(
                    [int(intersection_cam_another), int(union_cam_another)]
                )

            # Cluster masks with annotation masks
            for i in range(num_clusters):
                # Hard cluster mask with carcinoma
                if has_carcinoma:
                    intersection_cluster_carcinoma = (masks[i] * carcinoma_mask).sum()
                    union_cluster_carcinoma = (masks[i] + carcinoma_mask).sum()
                else:
                    intersection_cluster_carcinoma = 0
                    union_cluster_carcinoma = 0
                tile_data.extend(
                    [int(intersection_cluster_carcinoma), int(union_cluster_carcinoma)]
                )

                # Soft cluster mask with carcinoma
                if has_carcinoma:
                    intersection_soft_cluster_carcinoma = (
                        soft_masks[i] * carcinoma_mask
                    ).sum()
                    union_soft_cluster_carcinoma = (
                        soft_masks[i] + carcinoma_mask
                    ).sum()
                else:
                    intersection_soft_cluster_carcinoma = 0
                    union_soft_cluster_carcinoma = 0
                tile_data.extend(
                    [
                        int(intersection_soft_cluster_carcinoma),
                        int(union_soft_cluster_carcinoma),
                    ]
                )

                # Hard cluster mask with another_pathology
                if has_another_pathology:
                    intersection_cluster_another = (
                        masks[i] * another_pathology_mask
                    ).sum()
                    union_cluster_another = (masks[i] + another_pathology_mask).sum()
                else:
                    intersection_cluster_another = 0
                    union_cluster_another = 0
                tile_data.extend(
                    [int(intersection_cluster_another), int(union_cluster_another)]
                )

                # Soft cluster mask with another_pathology
                if has_another_pathology:
                    intersection_soft_cluster_another = (
                        soft_masks[i] * another_pathology_mask
                    ).sum()
                    union_soft_cluster_another = (
                        soft_masks[i] + another_pathology_mask
                    ).sum()
                else:
                    intersection_soft_cluster_another = 0
                    union_soft_cluster_another = 0
                tile_data.extend(
                    [
                        int(intersection_soft_cluster_another),
                        int(union_soft_cluster_another),
                    ]
                )

            assert len(tile_data) == len(header), (
                f"Expected {len(header)} columns, got {len(tile_data)}"
            )
            all_datasets[model_name].append(tile_data)

        hooked_model.zero_grad()

    # Convert each model's data list to DataFrame
    result = {}
    for model_name, data_list in all_datasets.items():
        _, num_clusters, clustering_algorithm = clustering_models[model_name]

        # Build header for this model
        header = ["label", "output", "clustering_algorithm", "num_clusters"]
        for i in range(num_clusters):
            header.append(f"weight_sum_{i}")
            header.append(f"cluster_count_{i}")
            header.append(f"cluster_soft_count_{i}")

        # Annotation mask presence flags
        header += ["has_carcinoma_mask", "has_another_pathology_mask"]

        for cam in ["grad_cam_pp", "grad_cam", "layer_cam"]:
            header += [f"{cam}_count"]
            for i in range(num_clusters):
                header += [f"intersection_count_{cam}_{i}", f"union_count_{cam}_{i}"]
                header += [
                    f"intersection_soft_count_{cam}_{i}",
                    f"union_soft_count_{cam}_{i}",
                ]
                header += [f"correlation_{cam}_{i}"]
                header += [f"l2_distance_{cam}_{i}"]

            # CAM with annotation masks
            header += [
                f"intersection_count_{cam}_carcinoma",
                f"union_count_{cam}_carcinoma",
                f"intersection_count_{cam}_another_pathology",
                f"union_count_{cam}_another_pathology",
            ]

        # Cluster masks with annotation masks
        for i in range(num_clusters):
            header += [
                f"intersection_count_cluster_{i}_carcinoma",
                f"union_count_cluster_{i}_carcinoma",
                f"intersection_soft_count_cluster_{i}_carcinoma",
                f"union_soft_count_cluster_{i}_carcinoma",
                f"intersection_count_cluster_{i}_another_pathology",
                f"union_count_cluster_{i}_another_pathology",
                f"intersection_soft_count_cluster_{i}_another_pathology",
                f"union_soft_count_cluster_{i}_another_pathology",
            ]

        result[model_name] = pd.DataFrame(data_list, columns=header)

    return result

In [12]:
def set_my_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

In [13]:
set_my_seed(42)

n = 1000

In [14]:
# First, generate the tile samples once to ensure we use the same tiles for all modelsprint("Generating tile samples (same tiles for all models)...")
#
num_slides = len(dataloaders)
slide_indices = np.random.choice(list(range(num_slides)), n, replace=True)
slide_counts = [0] * num_slides

for slide_index in slide_indices:
    slide_counts[slide_index] += 1

# Generate tile_samples: list of (slide_index, tile_idx) tuples

tile_samples = []
for slide_index, count in enumerate(slide_counts):
    if count == 0:
        continue
    dataloader = dataloaders[slide_index]
    dataset = dataloader.dataset
    num_tiles_in_slide = len(dataset)
    tile_indices = sorted(
        np.random.choice(list(range(num_tiles_in_slide)), count, replace=True)
    )
    for tile_idx in tile_indices:
        tile_samples.append((slide_index, tile_idx))


print(f"Generated {len(tile_samples)} tile samples")
print(
    "These same tiles will be used for all clustering algorithms and cluster numbers\n"
)

Generated 1000 tile samples
These same tiles will be used for all clustering algorithms and cluster numbers



In [15]:
# Create data for all models at once (same tiles, same activations)
print("Processing all tiles with all clustering models...")
all_datasets = create_data(clustering_models, tile_samples)

print(f"\nCreated {len(all_datasets)} separate datasets:")
for model_name, data in all_datasets.items():
    print(f"  {model_name}: {data.shape}")

# Save each dataset to a separate CSV file
print("\nSaving datasets...")
for _, data in all_datasets.items():
    filename = f"data_{model_name.lower()}.csv"
    data.to_csv(filename, index=False)
    print(f"  Saved {filename}")

# Also save combined data for convenience
combined_data = pd.concat(all_datasets.values(), ignore_index=True)
combined_filename = "data_all_models.csv"
combined_data.to_csv(combined_filename, index=False)
print(f"\nAlso saved combined data to {combined_filename}")
print(f"Combined data shape: {combined_data.shape}")

Processing all tiles with all clustering models...
Slide 0, Tile 96
818 index is 96
Label is tensor([0.])


/home/jovyan/prostate-cancer/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Slide 0, Tile 117
818 index is 117
Label is tensor([0.])
Slide 0, Tile 156
818 index is 156
Label is tensor([0.])
Slide 0, Tile 158
818 index is 158
Label is tensor([0.])
Slide 0, Tile 195
818 index is 195
Label is tensor([0.])
Slide 0, Tile 230
818 index is 230
Label is tensor([0.])
Slide 0, Tile 253
818 index is 253
Label is tensor([0.])
Slide 0, Tile 288
818 index is 288
Label is tensor([0.])
Slide 0, Tile 410
818 index is 410
Label is tensor([0.])
Slide 0, Tile 419
818 index is 419
Label is tensor([0.])
Slide 0, Tile 440
818 index is 440
Label is tensor([0.])
Slide 0, Tile 477
818 index is 477
Label is tensor([0.])
Slide 0, Tile 516
818 index is 516
Label is tensor([0.])
Slide 0, Tile 521
818 index is 521
Label is tensor([0.])
Slide 0, Tile 549
818 index is 549
Label is tensor([0.])
Slide 0, Tile 597
818 index is 597
Label is tensor([0.])
Slide 0, Tile 634
818 index is 634
Label is tensor([0.])
Slide 0, Tile 805
818 index is 805
Label is tensor([0.])
Slide 1, Tile 10
1095 index is 

ValueError: operands could not be broadcast together with shapes (1,32,32) (32,0) 

In [ ]:
# Load all separate datasets
all_datasets = {}

print("Loading separate datasets:")
for clustering_algorithm in CLUSTERING_ALGORITHMS:
    for num_clusters in CLUSTER_NUMBERS:
        model_name = f"{clustering_algorithm}_{num_clusters}"
        filename = f"data_{model_name.lower()}.csv"
        data = pd.read_csv(filename)
        all_datasets[model_name] = data
        print(f"  {model_name}: {data.shape}")

print(f"\nTotal datasets loaded: {len(all_datasets)}")

# Create combinations list from the datasets
combinations = []
for _, data in all_datasets.items():
    alg = data["clustering_algorithm"].iloc[0]
    n_clust = data["num_clusters"].iloc[0]
    combinations.append({"clustering_algorithm": alg, "num_clusters": n_clust})

combinations = pd.DataFrame(combinations).drop_duplicates()
print("\nAvailable combinations:")
print(combinations.to_string(index=False))

Loading separate datasets:
  NMF_4: (100, 66)
  NMF_6: (100, 96)
  NMF_8: (100, 126)
  NMF_10: (100, 156)
  NMF_12: (100, 186)
  NMF_14: (100, 216)
  KMeans_4: (100, 66)
  KMeans_6: (100, 96)
  KMeans_8: (100, 126)
  KMeans_10: (100, 156)
  KMeans_12: (100, 186)
  KMeans_14: (100, 216)

Total datasets loaded: 12

Available combinations:
clustering_algorithm  num_clusters
                 NMF             4
                 NMF             6
                 NMF             8
                 NMF            10
                 NMF            12
                 NMF            14
              KMeans             4
              KMeans             6
              KMeans             8
              KMeans            10
              KMeans            12
              KMeans            14


In [ ]:
base_figures_dir = "figures"
os.makedirs(base_figures_dir, exist_ok=True)

# Process each clustering model combination separately
for _, row in combinations.iterrows():
    alg = row["clustering_algorithm"]
    n_clust = row["num_clusters"]
    model_name = f"{alg}_{n_clust}"

    # Create subdirectory for this combination
    figures_dir = f"{base_figures_dir}/{model_name}"
    os.makedirs(figures_dir, exist_ok=True)

    print(f"\n{'=' * 80}")
    print(f"Processing: {alg} with {n_clust} clusters")
    print(f"{'=' * 80}\n")

    # Get the dataset for this combination
    data = all_datasets[model_name].copy()

    num_clusters = n_clust

    # y is the sigmoid of output
    data["y"] = 1 / (1 + np.exp(-data["output"]))

    # print count of labels - either 0 or 1
    print("Label distribution:")
    display(data["label"].value_counts())

    plt.hist(data["y"])
    plt.title(
        f"Distribution of predicted probabilities (y) - {alg} ({n_clust} clusters)"
    )
    plt.xlabel("y")
    plt.ylabel("Frequency")
    filename = f"{figures_dir}/hist_predicted_prob.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Compute prediction metrics
    data["predicted_label"] = (data["y"] > 0.94).astype(int)

    print(f"\nPrediction Metrics - {alg} with {n_clust} clusters:")
    print("=" * 60)
    print(f"Number of samples: {len(data)}")
    display(pd.crosstab(data["predicted_label"], data["label"]))
    print(f"Accuracy: {accuracy_score(data['label'], data['predicted_label'])}")
    print(f"Precision: {precision_score(data['label'], data['predicted_label'])}")
    print(f"Recall: {recall_score(data['label'], data['predicted_label'])}")
    print(f"F1 Score: {f1_score(data['label'], data['predicted_label'])}")
    print(f"AUC Score: {roc_auc_score(data['label'], data['y'])}")

    # Add IoU columns for each method and cluster
    for method in ["grad_cam_pp", "grad_cam", "layer_cam"]:
        for i in range(num_clusters):
            intersection_col = f"intersection_count_{method}_{i}"
            union_col = f"union_count_{method}_{i}"
            iou_col = f"iou_{method}_{i}"

            # Avoid division by zero; if union is zero, set IoU to np.nan
            mask = data[union_col].notna() & (data[union_col] > 0)
            data.loc[mask, iou_col] = (
                data.loc[mask, intersection_col] / data.loc[mask, union_col]
            )

    # Fit linear regression model
    print(f"\nFitting linear regression model - {alg} ({n_clust} clusters):")
    print("=" * 60)

    X = data[[f"weight_sum_{i}" for i in range(num_clusters)]]
    y = data["output"]

    model = LinearRegression()
    model.fit(X, y)

    print(f"Coefficients: {model.coef_}")
    print(f"Intercept: {model.intercept_}")
    print(f"R-squared: {model.score(X, y)}")

    # Fit linear regression model using cluster counts
    print(
        f"\nFitting linear regression model (using cluster counts) - {alg} ({n_clust} clusters):"
    )
    print("=" * 60)

    X_clusters = data[[f"cluster_count_{i}" for i in range(num_clusters)]]
    y = data["output"]

    model_clusters = LinearRegression()
    model_clusters.fit(X_clusters, y)

    print(f"Coefficients: {model_clusters.coef_}")
    print(f"Intercept: {model_clusters.intercept_}")
    print(f"R-squared: {model_clusters.score(X_clusters, y)}")

    # Visualize weight distributions by cluster
    weight_cols = [f"weight_sum_{i}" for i in range(num_clusters)]
    plt.figure(figsize=(14, 6))

    cluster_data_label0 = []
    cluster_data_label1 = []
    cluster_labels = []

    for i, weight_col in enumerate(weight_cols):
        weight_data_0 = data[data["predicted_label"] == 0][weight_col].dropna()
        weight_data_1 = data[data["predicted_label"] == 1][weight_col].dropna()

        cluster_data_label0.append(weight_data_0)
        cluster_data_label1.append(weight_data_1)
        cluster_labels.append(f"Cluster {i}")

    positions = np.arange(len(cluster_labels))
    width = 0.35

    bp0 = plt.boxplot(
        cluster_data_label0,
        positions=positions - width / 2,
        widths=width,
        tick_labels=cluster_labels,
        patch_artist=True,
    )
    bp1 = plt.boxplot(
        cluster_data_label1,
        positions=positions + width / 2,
        widths=width,
        patch_artist=True,
    )

    for patch in bp0["boxes"]:
        patch.set_facecolor("lightblue")
    for patch in bp1["boxes"]:
        patch.set_facecolor("lightcoral")

    plt.title(f"Weight Sum by Cluster - {alg} ({n_clust} clusters)")
    plt.xlabel("Cluster")
    plt.ylabel("Weight Sum")
    plt.xticks(positions, cluster_labels, rotation=45)
    plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
    plt.tight_layout()
    filename = f"{figures_dir}/weight_sum_by_label.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Weight sum plot (all data)
    plt.figure(figsize=(12, 6))
    cluster_data = []
    cluster_labels = []

    for i, weight_col in enumerate(weight_cols):
        weight_data = data[weight_col].dropna()
        cluster_data.append(weight_data)
        cluster_labels.append(f"Cluster {i}")

    plt.boxplot(cluster_data, tick_labels=cluster_labels)
    plt.title(f"Weight Sum by Cluster (All Data) - {alg} ({n_clust} clusters)")
    plt.xlabel("Cluster")
    plt.ylabel("Weight Sum")
    plt.xticks(rotation=45)
    plt.tight_layout()
    filename = f"{figures_dir}/weight_sum_all_data.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Visualize cluster count distributions by cluster
    count_cols = [f"cluster_count_{i}" for i in range(num_clusters)]
    plt.figure(figsize=(14, 6))

    cluster_data_label0 = []
    cluster_data_label1 = []
    cluster_labels = []

    for i, count_col in enumerate(count_cols):
        count_data_0 = data[data["predicted_label"] == 0][count_col].dropna()
        count_data_1 = data[data["predicted_label"] == 1][count_col].dropna()

        cluster_data_label0.append(count_data_0)
        cluster_data_label1.append(count_data_1)
        cluster_labels.append(f"Cluster {i}")

    positions = np.arange(len(cluster_labels))
    width = 0.35

    bp0 = plt.boxplot(
        cluster_data_label0,
        positions=positions - width / 2,
        widths=width,
        tick_labels=cluster_labels,
        patch_artist=True,
    )
    bp1 = plt.boxplot(
        cluster_data_label1,
        positions=positions + width / 2,
        widths=width,
        patch_artist=True,
    )

    for patch in bp0["boxes"]:
        patch.set_facecolor("lightblue")
    for patch in bp1["boxes"]:
        patch.set_facecolor("lightcoral")

    plt.title(f"Cluster Count by Cluster - {alg} ({n_clust} clusters)")
    plt.xlabel("Cluster")
    plt.ylabel("Cluster Count")
    plt.xticks(positions, cluster_labels, rotation=45)
    plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
    plt.tight_layout()
    filename = f"{figures_dir}/cluster_count_by_label.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Cluster count plot (all data)
    plt.figure(figsize=(12, 6))
    cluster_data = []
    cluster_labels = []

    for i, count_col in enumerate(count_cols):
        count_data = data[count_col].dropna()
        cluster_data.append(count_data)
        cluster_labels.append(f"Cluster {i}")

    plt.boxplot(cluster_data, tick_labels=cluster_labels)
    plt.title(f"Cluster Count by Cluster (All Data) - {alg} ({n_clust} clusters)")
    plt.xlabel("Cluster")
    plt.ylabel("Cluster Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    filename = f"{figures_dir}/cluster_count_all_data.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {filename}")

    # Add soft IoU columns for each method and cluster
    for method in ["grad_cam_pp", "grad_cam", "layer_cam"]:
        for i in range(num_clusters):
            intersection_col = f"intersection_soft_count_{method}_{i}"
            union_col = f"union_soft_count_{method}_{i}"
            iou_col = f"iou_soft_{method}_{i}"

            # Avoid division by zero; if union is zero, set IoU to np.nan
            mask = data[union_col].notna() & (data[union_col] > 0)
            data.loc[mask, iou_col] = (
                data.loc[mask, intersection_col] / data.loc[mask, union_col]
            )

    # Add IoU columns for CAMs with annotation masks
    for method in ["grad_cam_pp", "grad_cam", "layer_cam"]:
        # CAM with carcinoma
        intersection_col = f"intersection_count_{method}_carcinoma"
        union_col = f"union_count_{method}_carcinoma"
        iou_col = f"iou_{method}_carcinoma"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

        # CAM with another_pathology
        intersection_col = f"intersection_count_{method}_another_pathology"
        union_col = f"union_count_{method}_another_pathology"
        iou_col = f"iou_{method}_another_pathology"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

    # Add IoU columns for clusters with annotation masks
    for i in range(num_clusters):
        # Hard cluster mask with carcinoma
        intersection_col = f"intersection_count_cluster_{i}_carcinoma"
        union_col = f"union_count_cluster_{i}_carcinoma"
        iou_col = f"iou_cluster_{i}_carcinoma"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

        # Soft cluster mask with carcinoma
        intersection_col = f"intersection_soft_count_cluster_{i}_carcinoma"
        union_col = f"union_soft_count_cluster_{i}_carcinoma"
        iou_col = f"iou_soft_cluster_{i}_carcinoma"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

        # Hard cluster mask with another_pathology
        intersection_col = f"intersection_count_cluster_{i}_another_pathology"
        union_col = f"union_count_cluster_{i}_another_pathology"
        iou_col = f"iou_cluster_{i}_another_pathology"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

        # Soft cluster mask with another_pathology
        intersection_col = f"intersection_soft_count_cluster_{i}_another_pathology"
        union_col = f"union_soft_count_cluster_{i}_another_pathology"
        iou_col = f"iou_soft_cluster_{i}_another_pathology"
        mask = data[union_col].notna() & (data[union_col] > 0)
        data.loc[mask, iou_col] = (
            data.loc[mask, intersection_col] / data.loc[mask, union_col]
        )

    # Visualize for both Grad-CAM and Grad-CAM++
    for method in ["grad_cam", "grad_cam_pp", "layer_cam"]:
        method_display = (
            "Grad-CAM"
            if method == "grad_cam"
            else ("Grad-CAM++" if method == "grad_cam_pp" else "Layer-CAM")
        )

        # Visualize Soft IoU by cluster
        iou_cols = [f"iou_soft_{method}_{i}" for i in range(num_clusters)]

        plt.figure(figsize=(14, 6))

        cluster_data_label0 = []
        cluster_data_label1 = []
        cluster_labels = []

        for i, iou_col in enumerate(iou_cols):
            iou_data_0 = data[data["predicted_label"] == 0][iou_col].dropna()
            iou_data_1 = data[data["predicted_label"] == 1][iou_col].dropna()

            cluster_data_label0.append(iou_data_0)
            cluster_data_label1.append(iou_data_1)
            cluster_labels.append(f"Cluster {i}")

        positions = np.arange(len(cluster_labels))
        width = 0.35

        bp0 = plt.boxplot(
            cluster_data_label0,
            positions=positions - width / 2,
            widths=width,
            tick_labels=cluster_labels,
            patch_artist=True,
        )
        bp1 = plt.boxplot(
            cluster_data_label1,
            positions=positions + width / 2,
            widths=width,
            patch_artist=True,
        )

        for patch in bp0["boxes"]:
            patch.set_facecolor("lightblue")
        for patch in bp1["boxes"]:
            patch.set_facecolor("lightcoral")

        plt.title(f"{method_display} Soft IoU by Cluster - {alg} ({n_clust} clusters)")
        plt.xlabel("Cluster")
        plt.ylabel("Soft IoU")
        plt.xticks(positions, cluster_labels, rotation=45)
        plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
        plt.tight_layout()
        filename = f"{figures_dir}/soft_iou_by_label_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # Visualize Correlation by cluster
        correlation_cols = [f"correlation_{method}_{i}" for i in range(num_clusters)]

        plt.figure(figsize=(14, 6))

        cluster_data_label0 = []
        cluster_data_label1 = []
        cluster_labels = []

        for i, correlation_col in enumerate(correlation_cols):
            correlation_data_0 = data[data["predicted_label"] == 0][
                correlation_col
            ].dropna()
            correlation_data_1 = data[data["predicted_label"] == 1][
                correlation_col
            ].dropna()

            cluster_data_label0.append(correlation_data_0)
            cluster_data_label1.append(correlation_data_1)
            cluster_labels.append(f"Cluster {i}")

        positions = np.arange(len(cluster_labels))
        width = 0.35

        bp0 = plt.boxplot(
            cluster_data_label0,
            positions=positions - width / 2,
            widths=width,
            tick_labels=cluster_labels,
            patch_artist=True,
        )
        bp1 = plt.boxplot(
            cluster_data_label1,
            positions=positions + width / 2,
            widths=width,
            patch_artist=True,
        )

        for patch in bp0["boxes"]:
            patch.set_facecolor("lightblue")
        for patch in bp1["boxes"]:
            patch.set_facecolor("lightcoral")

        plt.title(
            f"{method_display} Correlation by Cluster - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("Correlation")
        plt.xticks(positions, cluster_labels, rotation=45)
        plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
        plt.tight_layout()
        filename = f"{figures_dir}/correlation_by_label_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # Visualize IoU by cluster (by label)
        iou_cols = [f"iou_{method}_{i}" for i in range(num_clusters)]

        plt.figure(figsize=(14, 6))

        cluster_data_label0 = []
        cluster_data_label1 = []
        cluster_labels = []

        for i, iou_col in enumerate(iou_cols):
            iou_data_0 = data[data["predicted_label"] == 0][iou_col].dropna()
            iou_data_1 = data[data["predicted_label"] == 1][iou_col].dropna()

            cluster_data_label0.append(iou_data_0)
            cluster_data_label1.append(iou_data_1)
            cluster_labels.append(f"Cluster {i}")

        positions = np.arange(len(cluster_labels))
        width = 0.35

        bp0 = plt.boxplot(
            cluster_data_label0,
            positions=positions - width / 2,
            widths=width,
            tick_labels=cluster_labels,
            patch_artist=True,
        )
        bp1 = plt.boxplot(
            cluster_data_label1,
            positions=positions + width / 2,
            widths=width,
            patch_artist=True,
        )

        for patch in bp0["boxes"]:
            patch.set_facecolor("lightblue")
        for patch in bp1["boxes"]:
            patch.set_facecolor("lightcoral")

        plt.title(f"{method_display} IoU by Cluster - {alg} ({n_clust} clusters)")
        plt.xlabel("Cluster")
        plt.ylabel("IoU")
        plt.xticks(positions, cluster_labels, rotation=45)
        plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
        plt.tight_layout()
        filename = f"{figures_dir}/iou_by_label_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # Create plots without aggregating by predicted label - all data together
        # 1. IoU plot
        iou_cols = [f"iou_{method}_{i}" for i in range(num_clusters)]
        plt.figure(figsize=(12, 6))
        cluster_data = []
        cluster_labels = []

        for i, iou_col in enumerate(iou_cols):
            iou_data = data[iou_col].dropna()
            cluster_data.append(iou_data)
            cluster_labels.append(f"Cluster {i}")

        plt.boxplot(cluster_data, tick_labels=cluster_labels)
        plt.title(
            f"{method_display} IoU by Cluster (All Data) - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("IoU")
        plt.xticks(rotation=45)
        plt.tight_layout()
        filename = f"{figures_dir}/iou_all_data_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # 2. Soft IoU plot
        soft_iou_cols = [f"iou_soft_{method}_{i}" for i in range(num_clusters)]
        plt.figure(figsize=(12, 6))
        cluster_data = []
        cluster_labels = []

        for i, iou_col in enumerate(soft_iou_cols):
            iou_data = data[iou_col].dropna()
            cluster_data.append(iou_data)
            cluster_labels.append(f"Cluster {i}")

        plt.boxplot(cluster_data, tick_labels=cluster_labels)
        plt.title(
            f"{method_display} Soft IoU by Cluster (All Data) - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("Soft IoU")
        plt.xticks(rotation=45)
        plt.tight_layout()
        filename = f"{figures_dir}/soft_iou_all_data_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # 3. Correlation plot
        correlation_cols = [f"correlation_{method}_{i}" for i in range(num_clusters)]
        plt.figure(figsize=(12, 6))
        cluster_data = []
        cluster_labels = []

        for i, correlation_col in enumerate(correlation_cols):
            correlation_data = data[correlation_col].dropna()
            cluster_data.append(correlation_data)
            cluster_labels.append(f"Cluster {i}")

        plt.boxplot(cluster_data, tick_labels=cluster_labels)
        plt.title(
            f"{method_display} Correlation by Cluster (All Data) - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("Correlation")
        plt.xticks(rotation=45)
        plt.tight_layout()
        filename = f"{figures_dir}/correlation_all_data_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # 4. L2 Distance plot (by label)
        l2_cols = [f"l2_distance_{method}_{i}" for i in range(num_clusters)]
        plt.figure(figsize=(14, 6))

        cluster_data_label0 = []
        cluster_data_label1 = []
        cluster_labels = []

        for i, l2_col in enumerate(l2_cols):
            l2_data_0 = data[data["predicted_label"] == 0][l2_col].dropna()
            l2_data_1 = data[data["predicted_label"] == 1][l2_col].dropna()

            cluster_data_label0.append(l2_data_0)
            cluster_data_label1.append(l2_data_1)
            cluster_labels.append(f"Cluster {i}")

        positions = np.arange(len(cluster_labels))
        width = 0.35

        bp0 = plt.boxplot(
            cluster_data_label0,
            positions=positions - width / 2,
            widths=width,
            tick_labels=cluster_labels,
            patch_artist=True,
        )
        bp1 = plt.boxplot(
            cluster_data_label1,
            positions=positions + width / 2,
            widths=width,
            patch_artist=True,
        )

        for patch in bp0["boxes"]:
            patch.set_facecolor("lightblue")
        for patch in bp1["boxes"]:
            patch.set_facecolor("lightcoral")

        plt.title(
            f"{method_display} L2 Distance by Cluster - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("L2 Distance")
        plt.xticks(positions, cluster_labels, rotation=45)
        plt.legend([bp0["boxes"][0], bp1["boxes"][0]], ["Label 0", "Label 1"])
        plt.tight_layout()
        filename = f"{figures_dir}/l2_distance_by_label_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

        # 5. L2 Distance plot (all data)
        l2_cols = [f"l2_distance_{method}_{i}" for i in range(num_clusters)]
        plt.figure(figsize=(12, 6))
        cluster_data = []
        cluster_labels = []

        for i, l2_col in enumerate(l2_cols):
            l2_data = data[l2_col].dropna()
            cluster_data.append(l2_data)
            cluster_labels.append(f"Cluster {i}")

        plt.boxplot(cluster_data, tick_labels=cluster_labels)
        plt.title(
            f"{method_display} L2 Distance by Cluster (All Data) - {alg} ({n_clust} clusters)"
        )
        plt.xlabel("Cluster")
        plt.ylabel("L2 Distance")
        plt.xticks(rotation=45)
        plt.tight_layout()
        filename = f"{figures_dir}/l2_distance_all_data_{method}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved figure: {filename}")

    # Visualize CAM IoU with annotation masks
    for method in ["grad_cam", "grad_cam_pp", "layer_cam"]:
        method_display = (
            "Grad-CAM"
            if method == "grad_cam"
            else ("Grad-CAM++" if method == "grad_cam_pp" else "Layer-CAM")
        )

        # Filter data to only include samples with masks
        data_with_carcinoma = data[data["has_carcinoma_mask"] == 1]
        data_with_another = data[data["has_another_pathology_mask"] == 1]

        # CAM IoU with carcinoma
        if len(data_with_carcinoma) > 0:
            iou_col = f"iou_{method}_carcinoma"
            iou_data = data_with_carcinoma[iou_col].dropna()
            if len(iou_data) > 0:
                plt.figure(figsize=(10, 6))
                plt.boxplot([iou_data], labels=[f"{method_display} with Carcinoma"])
                plt.title(
                    f"{method_display} IoU with Carcinoma Mask - {alg} ({n_clust} clusters)"
                )
                plt.ylabel("IoU")
                plt.tight_layout()
                filename = f"{figures_dir}/iou_{method}_carcinoma.png"
                plt.savefig(filename, dpi=300, bbox_inches="tight")
                plt.close()
                print(f"Saved figure: {filename}")

        # CAM IoU with another_pathology
        if len(data_with_another) > 0:
            iou_col = f"iou_{method}_another_pathology"
            iou_data = data_with_another[iou_col].dropna()
            if len(iou_data) > 0:
                plt.figure(figsize=(10, 6))
                plt.boxplot(
                    [iou_data], labels=[f"{method_display} with Another Pathology"]
                )
                plt.title(
                    f"{method_display} IoU with Another Pathology Mask - {alg} ({n_clust} clusters)"
                )
                plt.ylabel("IoU")
                plt.tight_layout()
                filename = f"{figures_dir}/iou_{method}_another_pathology.png"
                plt.savefig(filename, dpi=300, bbox_inches="tight")
                plt.close()
                print(f"Saved figure: {filename}")

    # Visualize Cluster IoU with annotation masks
    data_with_carcinoma = data[data["has_carcinoma_mask"] == 1]
    data_with_another = data[data["has_another_pathology_mask"] == 1]

    # Cluster IoU with carcinoma (hard and soft)
    if len(data_with_carcinoma) > 0:
        iou_hard_cols = [f"iou_cluster_{i}_carcinoma" for i in range(num_clusters)]
        iou_soft_cols = [f"iou_soft_cluster_{i}_carcinoma" for i in range(num_clusters)]

        # Hard masks
        cluster_data = []
        cluster_labels = []
        for i, iou_col in enumerate(iou_hard_cols):
            iou_data = data_with_carcinoma[iou_col].dropna()
            if len(iou_data) > 0:
                cluster_data.append(iou_data)
                cluster_labels.append(f"Cluster {i}")

        if len(cluster_data) > 0:
            plt.figure(figsize=(12, 6))
            plt.boxplot(cluster_data, tick_labels=cluster_labels)
            plt.title(
                f"Cluster IoU (Hard) with Carcinoma Mask - {alg} ({n_clust} clusters)"
            )
            plt.xlabel("Cluster")
            plt.ylabel("IoU")
            plt.xticks(rotation=45)
            plt.tight_layout()
            filename = f"{figures_dir}/iou_cluster_hard_carcinoma.png"
            plt.savefig(filename, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved figure: {filename}")

        # Soft masks
        cluster_data = []
        cluster_labels = []
        for i, iou_col in enumerate(iou_soft_cols):
            iou_data = data_with_carcinoma[iou_col].dropna()
            if len(iou_data) > 0:
                cluster_data.append(iou_data)
                cluster_labels.append(f"Cluster {i}")

        if len(cluster_data) > 0:
            plt.figure(figsize=(12, 6))
            plt.boxplot(cluster_data, tick_labels=cluster_labels)
            plt.title(
                f"Cluster IoU (Soft) with Carcinoma Mask - {alg} ({n_clust} clusters)"
            )
            plt.xlabel("Cluster")
            plt.ylabel("IoU")
            plt.xticks(rotation=45)
            plt.tight_layout()
            filename = f"{figures_dir}/iou_cluster_soft_carcinoma.png"
            plt.savefig(filename, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved figure: {filename}")

    # Cluster IoU with another_pathology (hard and soft)
    if len(data_with_another) > 0:
        iou_hard_cols = [
            f"iou_cluster_{i}_another_pathology" for i in range(num_clusters)
        ]
        iou_soft_cols = [
            f"iou_soft_cluster_{i}_another_pathology" for i in range(num_clusters)
        ]

        # Hard masks
        cluster_data = []
        cluster_labels = []
        for i, iou_col in enumerate(iou_hard_cols):
            iou_data = data_with_another[iou_col].dropna()
            if len(iou_data) > 0:
                cluster_data.append(iou_data)
                cluster_labels.append(f"Cluster {i}")

        if len(cluster_data) > 0:
            plt.figure(figsize=(12, 6))
            plt.boxplot(cluster_data, tick_labels=cluster_labels)
            plt.title(
                f"Cluster IoU (Hard) with Another Pathology Mask - {alg} ({n_clust} clusters)"
            )
            plt.xlabel("Cluster")
            plt.ylabel("IoU")
            plt.xticks(rotation=45)
            plt.tight_layout()
            filename = f"{figures_dir}/iou_cluster_hard_another_pathology.png"
            plt.savefig(filename, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved figure: {filename}")

        # Soft masks
        cluster_data = []
        cluster_labels = []
        for i, iou_col in enumerate(iou_soft_cols):
            iou_data = data_with_another[iou_col].dropna()
            if len(iou_data) > 0:
                cluster_data.append(iou_data)
                cluster_labels.append(f"Cluster {i}")

        if len(cluster_data) > 0:
            plt.figure(figsize=(12, 6))
            plt.boxplot(cluster_data, tick_labels=cluster_labels)
            plt.title(
                f"Cluster IoU (Soft) with Another Pathology Mask - {alg} ({n_clust} clusters)"
            )
            plt.xlabel("Cluster")
            plt.ylabel("IoU")
            plt.xticks(rotation=45)
            plt.tight_layout()
            filename = f"{figures_dir}/iou_cluster_soft_another_pathology.png"
            plt.savefig(filename, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved figure: {filename}")

    # Create tables split by CAM method and label
    print(f"\n{'=' * 60}")
    print(f"Statistics (Mean ± Std) - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")

    total_pixels = 32 * 32  # Total number of pixels

    # For each CAM method
    for method in ["grad_cam", "grad_cam_pp", "layer_cam"]:
        method_display = (
            "Grad-CAM"
            if method == "grad_cam"
            else ("Grad-CAM++" if method == "grad_cam_pp" else "Layer-CAM")
        )

        # For each label split (Overall, Label 0, Label 1)
        for label_filter in [None, 0, 1]:
            if label_filter is None:
                label_name = "Overall"
                label_data = data
            else:
                label_name = f"Label {label_filter}"
                label_data = data[data["predicted_label"] == label_filter]

            print(f"\n{method_display} - {label_name}:")
            print("-" * 60)

            # Build table for this method and label
            table_data = []
            for i in range(num_clusters):
                row = {"Cluster": i}

                # Weight Sum
                weight_col = f"weight_sum_{i}"
                weight_values = label_data[weight_col].dropna()
                if len(weight_values) > 0:
                    row["Weight Sum"] = (
                        f"{weight_values.mean():.4f} ± {weight_values.std():.4f}"
                    )
                else:
                    row["Weight Sum"] = "N/A"

                # Cluster Count
                count_col = f"cluster_count_{i}"
                count_values = label_data[count_col].dropna()
                if len(count_values) > 0:
                    row["Cluster Count"] = (
                        f"{count_values.mean():.4f} ± {count_values.std():.4f}"
                    )
                else:
                    row["Cluster Count"] = "N/A"

                # Normalized Cluster Count
                if len(count_values) > 0:
                    normalized_counts = count_values / total_pixels
                    row["Norm Count"] = (
                        f"{normalized_counts.mean():.4f} ± {normalized_counts.std():.4f}"
                    )
                else:
                    row["Norm Count"] = "N/A"

                # IoU
                iou_col = f"iou_{method}_{i}"
                iou_values = label_data[iou_col].dropna()
                if len(iou_values) > 0:
                    row["IoU"] = f"{iou_values.mean():.4f} ± {iou_values.std():.4f}"
                else:
                    row["IoU"] = "N/A"

                # Soft IoU
                soft_iou_col = f"iou_soft_{method}_{i}"
                soft_iou_values = label_data[soft_iou_col].dropna()
                if len(soft_iou_values) > 0:
                    row["Soft IoU"] = (
                        f"{soft_iou_values.mean():.4f} ± {soft_iou_values.std():.4f}"
                    )
                else:
                    row["Soft IoU"] = "N/A"

                # Correlation
                corr_col = f"correlation_{method}_{i}"
                corr_values = label_data[corr_col].dropna()
                if len(corr_values) > 0:
                    row["Correlation"] = (
                        f"{corr_values.mean():.4f} ± {corr_values.std():.4f}"
                    )
                else:
                    row["Correlation"] = "N/A"

                # L2 Distance
                l2_col = f"l2_distance_{method}_{i}"
                l2_values = label_data[l2_col].dropna()
                if len(l2_values) > 0:
                    row["L2 Distance"] = (
                        f"{l2_values.mean():.4f} ± {l2_values.std():.4f}"
                    )
                else:
                    row["L2 Distance"] = "N/A"

                table_data.append(row)

            if table_data:
                table_df = pd.DataFrame(table_data)
                display(table_df)

    # Print statistics for annotation mask metrics
    print(f"\n{'=' * 60}")
    print(f"Annotation Mask Statistics - {alg} ({n_clust} clusters)")
    print(f"{'=' * 60}\n")

    # CAM IoU with annotation masks
    data_with_carcinoma = data[data["has_carcinoma_mask"] == 1]
    data_with_another = data[data["has_another_pathology_mask"] == 1]

    if len(data_with_carcinoma) > 0:
        print(f"\nCAM IoU with Carcinoma Mask (n={len(data_with_carcinoma)}):")
        print("-" * 60)
        table_data = []
        for method in ["grad_cam", "grad_cam_pp", "layer_cam"]:
            method_display = (
                "Grad-CAM"
                if method == "grad_cam"
                else ("Grad-CAM++" if method == "grad_cam_pp" else "Layer-CAM")
            )
            iou_col = f"iou_{method}_carcinoma"
            iou_values = data_with_carcinoma[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Method": method_display,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

    if len(data_with_another) > 0:
        print(f"\nCAM IoU with Another Pathology Mask (n={len(data_with_another)}):")
        print("-" * 60)
        table_data = []
        for method in ["grad_cam", "grad_cam_pp", "layer_cam"]:
            method_display = (
                "Grad-CAM"
                if method == "grad_cam"
                else ("Grad-CAM++" if method == "grad_cam_pp" else "Layer-CAM")
            )
            iou_col = f"iou_{method}_another_pathology"
            iou_values = data_with_another[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Method": method_display,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

    # Cluster IoU with annotation masks
    if len(data_with_carcinoma) > 0:
        print(
            f"\nCluster IoU (Hard) with Carcinoma Mask (n={len(data_with_carcinoma)}):"
        )
        print("-" * 60)
        table_data = []
        for i in range(num_clusters):
            iou_col = f"iou_cluster_{i}_carcinoma"
            iou_values = data_with_carcinoma[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Cluster": i,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

        print(
            f"\nCluster IoU (Soft) with Carcinoma Mask (n={len(data_with_carcinoma)}):"
        )
        print("-" * 60)
        table_data = []
        for i in range(num_clusters):
            iou_col = f"iou_soft_cluster_{i}_carcinoma"
            iou_values = data_with_carcinoma[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Cluster": i,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

    if len(data_with_another) > 0:
        print(
            f"\nCluster IoU (Hard) with Another Pathology Mask (n={len(data_with_another)}):"
        )
        print("-" * 60)
        table_data = []
        for i in range(num_clusters):
            iou_col = f"iou_cluster_{i}_another_pathology"
            iou_values = data_with_another[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Cluster": i,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

        print(
            f"\nCluster IoU (Soft) with Another Pathology Mask (n={len(data_with_another)}):"
        )
        print("-" * 60)
        table_data = []
        for i in range(num_clusters):
            iou_col = f"iou_soft_cluster_{i}_another_pathology"
            iou_values = data_with_another[iou_col].dropna()
            if len(iou_values) > 0:
                table_data.append(
                    {
                        "Cluster": i,
                        "IoU (Mean ± Std)": f"{iou_values.mean():.4f} ± {iou_values.std():.4f}",
                    }
                )
        if table_data:
            table_df = pd.DataFrame(table_data)
            display(table_df)

    print(f"\n{'=' * 80}\n")


Processing: NMF with 4 clusters

Label distribution:


label
0.0    78
1.0    22
Name: count, dtype: int64

Saved figure: figures/NMF_4/hist_predicted_prob.png

Prediction Metrics - NMF with 4 clusters:
Number of samples: 100


label,0.0,1.0
predicted_label,,
0,78,3
1,0,19


Accuracy: 0.97
Precision: 1.0
Recall: 0.8636363636363636
F1 Score: 0.926829268292683
AUC Score: 0.9924242424242424

Fitting linear regression model - NMF (4 clusters):
Coefficients: [-0.04509821  0.06407089 -0.01535575  0.39810952]
Intercept: -2.8623224188080183
R-squared: 0.8457055995485877

Fitting linear regression model (using cluster counts) - NMF (4 clusters):
Coefficients: [-0.01114124  0.0150533  -0.01170208  0.00779003]
Intercept: 5.073764017462381
R-squared: 0.7249468116433635
Saved figure: figures/NMF_4/weight_sum_by_label.png
Saved figure: figures/NMF_4/weight_sum_all_data.png
Saved figure: figures/NMF_4/cluster_count_by_label.png
Saved figure: figures/NMF_4/cluster_count_all_data.png
Saved figure: figures/NMF_4/soft_iou_by_label_grad_cam.png
Saved figure: figures/NMF_4/correlation_by_label_grad_cam.png
Saved figure: figures/NMF_4/iou_by_label_grad_cam.png
Saved figure: figures/NMF_4/iou_all_data_grad_cam.png
Saved figure: figures/NMF_4/soft_iou_all_data_grad_cam.png
Saved 

...

,Cluster,Weight Sum,Cluster Count,Norm Count,IoU,Soft IoU,Correlation,L2 Distance
0,0,3022.8638 ± 1782.9042,604.2400 ± 228.2321,0.5901 ± 0.2229,0.1507 ± 0.1116,0.1415 ± 0.1385,-0.1572 ± 0.1912,16.1100 ± 5.1310
1,1,6245.0897 ± 1244.8774,9.7800 ± 17.6404,0.0096 ± 0.0172,0.0065 ± 0.0238,0.1415 ± 0.1385,0.2261 ± 0.2183,11.3504 ± 3.8391
2,2,6981.0914 ± 976.5995,30.3600 ± 40.3621,0.0296 ± 0.0394,0.0001 ± 0.0010,0.1415 ± 0.1385,0.1727 ± 0.2645,9.2840 ± 2.8666
3,3,3320.1796 ± 1610.9493,56.7800 ± 88.6367,0.0554 ± 0.0866,0.0069 ± 0.0120,0.1415 ± 0.1385,-0.1318 ± 0.2597,11.1192 ± 1.8769
4,4,3173.2014 ± 1657.6570,61.4800 ± 53.5706,0.0600 ± 0.0523,0.0139 ± 0.0201,0.1415 ± 0.1385,-0.1737 ± 0.2653,11.6942 ± 2.1116
5,5,5984.6026 ± 1105.3299,39.3900 ± 82.5529,0.0385 ± 0.0806,0.0643 ± 0.0889,0.1415 ± 0.1385,0.3274 ± 0.2526,12.3743 ± 3.8475
6,6,5173.8622 ± 1368.3926,22.0200 ± 26.3385,0.0215 ± 0.0257,0.0134 ± 0.0304,0.1415 ± 0.1385,0.1857 ± 0.1540,11.3400 ± 2.6500
7,7,4419.0197 ± 1230.4670,37.9800 ± 37.5752,0.0371 ± 0.0367,0.0016 ± 0.0054,0.1415 ± 0.1385,0.0851 ± 0.1362,10.9469 ± 2.6800
8,8,11252.7618 ± 1046.9475,19.7100 ± 33.3222,0.0192 ± 0.0325,0.0000 ± 0.0000,0.1415 ± 0.1385,0.2065 ± 0.3136,7.2077 ± 2.0188
9,9,7212.3348 ± 1010.4548,38.3900 ± 65.1559,0.0375 ± 0.0636,0.0000 ± 0.0000,0.1415 ± 0.1385,0.1671 ± 0.2651,8.7031 ± 2.6810



Grad-CAM++ - Label 0:
------------------------------------------------------------


,Cluster,Weight Sum,Cluster Count,Norm Count,IoU,Soft IoU,Correlation,L2 Distance
0,0,2774.9022 ± 1647.3551,588.4074 ± 240.8248,0.5746 ± 0.2352,0.1172 ± 0.0905,0.0855 ± 0.0759,-0.0839 ± 0.1265,16.5535 ± 5.3854
1,1,6054.1030 ± 1091.3935,10.5926 ± 18.4416,0.0103 ± 0.0180,0.0077 ± 0.0263,0.0855 ± 0.0759,0.1499 ± 0.1553,11.0317 ± 3.7126
2,2,6649.4722 ± 434.5373,36.5309 ± 42.3217,0.0357 ± 0.0413,0.0001 ± 0.0012,0.0855 ± 0.0759,0.0654 ± 0.1503,8.8693 ± 2.3157
3,3,3048.9456 ± 1416.3145,67.0617 ± 95.4088,0.0655 ± 0.0932,0.0076 ± 0.0131,0.0855 ± 0.0759,-0.0198 ± 0.1241,10.7935 ± 1.5947
4,4,2906.4487 ± 1473.4095,66.8642 ± 56.5910,0.0653 ± 0.0553,0.0146 ± 0.0220,0.0855 ± 0.0759,-0.0652 ± 0.1550,11.6212 ± 2.1268
5,5,5958.4696 ± 1123.7153,6.1111 ± 15.0449,0.0060 ± 0.0147,0.0294 ± 0.0493,0.0855 ± 0.0759,0.2573 ± 0.2230,12.8205 ± 4.0430
6,6,4964.6226 ± 1209.7093,23.8642 ± 26.9531,0.0233 ± 0.0263,0.0158 ± 0.0333,0.0855 ± 0.0759,0.1449 ± 0.1230,11.0354 ± 2.4076
7,7,4107.7056 ± 858.0111,45.1111 ± 37.9101,0.0441 ± 0.0370,0.0020 ± 0.0060,0.0855 ± 0.0759,0.0493 ± 0.1027,10.7133 ± 1.9718
8,8,10906.0692 ± 688.9855,24.0988 ± 35.6148,0.0235 ± 0.0348,0.0000 ± 0.0000,0.0855 ± 0.0759,0.0757 ± 0.1719,7.0034 ± 1.8146
9,9,6874.9550 ± 507.1200,46.2222 ± 70.0295,0.0451 ± 0.0684,0.0000 ± 0.0000,0.0855 ± 0.0759,0.0605 ± 0.1511,8.5141 ± 2.0486



Grad-CAM++ - Label 1:
------------------------------------------------------------


,Cluster,Weight Sum,Cluster Count,Norm Count,IoU,Soft IoU,Correlation,L2 Distance
0,0,4079.9637 ± 1991.3937,671.7368 ± 150.9797,0.6560 ± 0.1474,0.2938 ± 0.0736,0.3803 ± 0.0745,-0.4621 ± 0.0777,14.2196 ± 3.3532
1,1,7059.2962 ± 1538.0510,6.3158 ± 13.5772,0.0062 ± 0.0133,0.0012 ± 0.0027,0.3803 ± 0.0745,0.5428 ± 0.1446,12.7088 ± 4.1710
2,2,8394.8361 ± 1342.3161,4.0526 ± 9.8797,0.0040 ± 0.0096,0.0000 ± 0.0000,0.3803 ± 0.0745,0.6185 ± 0.1366,11.0518 ± 4.1582
3,3,4476.4929 ± 1901.4867,12.9474 ± 15.8025,0.0126 ± 0.0154,0.0040 ± 0.0052,0.3803 ± 0.0745,-0.5973 ± 0.1141,12.5078 ± 2.3582
4,4,4310.4102 ± 1944.0031,38.5263 ± 29.3908,0.0376 ± 0.0287,0.0108 ± 0.0074,0.3803 ± 0.0745,-0.6246 ± 0.0950,12.0057 ± 2.0725
5,5,6096.0114 ± 1044.6590,181.2632 ± 101.2252,0.1770 ± 0.0989,0.2093 ± 0.0673,0.3803 ± 0.0745,0.6188 ± 0.1313,10.4720 ± 2.0135
6,6,6065.8834 ± 1660.6723,14.1579 ± 22.4976,0.0138 ± 0.0220,0.0037 ± 0.0078,0.3803 ± 0.0745,0.3551 ± 0.1568,12.6386 ± 3.2653
7,7,5746.2010 ± 1661.3110,7.5789 ± 13.1754,0.0074 ± 0.0129,0.0004 ± 0.0007,0.3803 ± 0.0745,0.2338 ± 0.1592,11.9426 ± 4.5753
8,8,12730.7668 ± 1035.3849,1.0000 ± 3.6667,0.0010 ± 0.0036,0.0000 ± 0.0000,0.3803 ± 0.0745,0.7505 ± 0.1197,8.0788 ± 2.6062
9,9,8650.6381 ± 1332.9534,5.0000 ± 10.0056,0.0049 ± 0.0098,0.0000 ± 0.0000,0.3803 ± 0.0745,0.6105 ± 0.1501,9.5091 ± 4.4760
